In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import os
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer

In [2]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/himau23/preprocessed-question-pairs/preprossed(1).csv
/kaggle/input/datasets/quora/question-pairs-dataset/questions.csv


In [3]:
DATA_PATH = "/kaggle/input/datasets/quora/question-pairs-dataset/questions.csv"
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (404351, 6)
Columns: ['id', 'qid1', 'qid2', 'question1', 'question2', 'is_duplicate']


,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


In [4]:
MODEL_NAME = "BAAI/bge-small-en-v1.5"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)
model.eval()

print("Device:", device)

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Device: cuda


In [5]:
def get_embedding(text):
    encoded = tokenizer(
        text,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )
    
    encoded = {
        key: value.to(device)
        for key, value in encoded.items()
    }
    
    with torch.no_grad():
        outputs = model(**encoded)
        
    token_embeddings = outputs.last_hidden_state
    attention_mask = encoded["attention_mask"]
    
    mask = attention_mask.unsqueeze(-1).expand(
        token_embeddings.size()
    ).float()
    
    embedding = torch.sum(
        token_embeddings * mask,
        dim=1
    ) / torch.clamp(
        mask.sum(dim=1),
        min=1e-9
    )
    
    embedding = F.normalize(
        embedding,
        p=2,
        dim=1
    )
    
    return embedding

In [6]:
q1 = df.loc[0, "question1"]
q2 = df.loc[0, "question2"]

print("Question 1:", q1)
print("Question 2:", q2)
print("Label:", df.loc[0, "is_duplicate"])

e1 = get_embedding(q1)
e2 = get_embedding(q2)

print("Embedding 1:", e1.shape)
print("Embedding 2:", e2.shape)

similarity = torch.sum(e1 * e2).item()
print("BGE Semantic Similarity:", similarity)

Question 1: What is the step by step guide to invest in share market in india?
Question 2: What is the step by step guide to invest in share market?
Label: 0
Embedding 1: torch.Size([1, 384])
Embedding 2: torch.Size([1, 384])
BGE Semantic Similarity: 0.920744776725769


In [7]:
bge_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5",
    device="cuda"
)

questions1 = df["question1"].fillna("").astype(str).tolist()
questions2 = df["question2"].fillna("").astype(str).tolist()

embeddings_q1 = bge_model.encode(
    questions1,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings_q2 = bge_model.encode(
    questions2,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/6318 [00:00<?, ?it/s]

Batches:   0%|          | 0/6318 [00:00<?, ?it/s]

In [8]:
bge_similarity = np.sum(
    embeddings_q1 * embeddings_q2,
    axis=1
)

df["bge_similarity"] = bge_similarity
print("Q1 embeddings:", embeddings_q1.shape)
print("Q2 embeddings:", embeddings_q2.shape)
df[["question1", "question2", "is_duplicate", "bge_similarity"]].head(10)

Q1 embeddings: (404351, 384)
Q2 embeddings: (404351, 384)


,question1,question2,is_duplicate,bge_similarity
0,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0,0.922287
1,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0,0.693393
2,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0,0.826137
3,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0,0.505303
4,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0,0.552441
5,Astrology: I am a Capricorn Sun Cap moon and c...,"I'm a triple Capricorn (Sun, Moon and ascendan...",1,0.858212
6,Should I buy tiago?,What keeps childern active and far from phone ...,0,0.434120
7,How can I be a good geologist?,What should I do to be a great geologist?,1,0.928560
8,When do you use シ instead of し?,"When do you use ""&"" instead of ""and""?",0,0.700380
9,Motorola (company): Can I hack my Charter Moto...,How do I hack Motorola DCX3400 for free internet?,0,0.835582


In [9]:
df.to_csv("/kaggle/working/embedded_questions_quora.csv", index=False)

In [10]:
df.head()

,id,qid1,qid2,question1,question2,is_duplicate,bge_similarity
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0,0.922287
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0,0.693393
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0,0.826137
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0,0.505303
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0,0.552441


# Combining BGE embeddings with Preprocessed to compare the accuracy of the model with other.

In [11]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/himau23/quora-preprocessed-dataset/preprossed(1).csv
/kaggle/input/datasets/quora/question-pairs-dataset/questions.csv


In [12]:
DATA_PATH = "/kaggle/input/datasets/himau23/quora-preprocessed-dataset/preprossed(1).csv"
prep_df = pd.read_csv(DATA_PATH)

print("Shape:", prep_df.shape)
print("Columns:", prep_df.columns.tolist())
prep_df.head()

Shape: (50000, 29)
Columns: ['Unnamed: 0', 'id', 'qid1', 'qid2', 'question1', 'question2', 'is_duplicate', 'q1_len', 'q2_len', 'q1_num_words', 'q2_num_words', 'word_common', 'word_total', 'word_share', 'cwc_min', 'cwc_max', 'csc_min', 'csc_max', 'ctc_min', 'ctc_max', 'last_word_eq', 'first_word_eq', 'abs_len_diff', 'mean_len', 'longest_substr_ratio', 'fuzz_ratio', 'fuzz_partial_ratio', 'token_sort_ratio', 'token_set_ratio']


,Unnamed: 0,id,qid1,qid2,question1,question2,is_duplicate,q1_len,q2_len,q1_num_words,...,ctc_max,last_word_eq,first_word_eq,abs_len_diff,mean_len,longest_substr_ratio,fuzz_ratio,fuzz_partial_ratio,token_sort_ratio,token_set_ratio
0,339499,339499,665522,665523,why was cyrus mistry removed as the chairman o...,why did the tata sons sacked cyrus mistry,1,57,41,11,...,0.545450,0.0,1.0,3.0,9.5,0.309524,39,46,67,85
1,289521,289521,568878,568879,by what age would you think a man should be ma...,when my wrist is extended i feel a shock and b...,0,51,104,11,...,0.136363,0.0,0.0,11.0,16.5,0.115385,26,50,35,37
2,4665,4665,9325,9326,how would an arbitrageur seek to capitalize gi...,how would an arbitrageur seek to capitalize gi...,0,123,122,42,...,0.708330,1.0,1.0,0.0,24.0,0.593496,100,99,100,99
3,54203,54203,107861,107862,why did quora mark my question as incomplete,why does quora detect my question as an incomp...,1,44,59,8,...,0.599994,0.0,1.0,2.0,9.0,0.355556,74,75,74,89
4,132566,132566,262554,91499,what is it like working with pivotal labs as a...,what is it like to work at pivotal labs,0,53,39,11,...,0.545450,0.0,1.0,2.0,10.0,0.400000,76,85,78,84


In [13]:
print("Shape:", prep_df.shape)

print("\nMissing values:")
print(prep_df.isna().sum().sum())

print("\nDuplicate IDs:")
print(prep_df['id'].duplicated().sum())

print("\nTarget distribution:")
print(prep_df['is_duplicate'].value_counts())

print("\nTarget proportions:")
print(prep_df['is_duplicate'].value_counts(normalize=True))

Shape: (50000, 29)

Missing values:
2

Duplicate IDs:
0

Target distribution:
is_duplicate
0    31594
1    18406
Name: count, dtype: int64

Target proportions:
is_duplicate
0    0.63188
1    0.36812
Name: proportion, dtype: float64


In [14]:
missing = prep_df.isna().sum()

print(missing[missing > 0])

question1    1
question2    1
dtype: int64


In [15]:
prep_df[prep_df.isna().any(axis=1)]

,Unnamed: 0,id,qid1,qid2,question1,question2,is_duplicate,q1_len,q2_len,q1_num_words,...,ctc_max,last_word_eq,first_word_eq,abs_len_diff,mean_len,longest_substr_ratio,fuzz_ratio,fuzz_partial_ratio,token_sort_ratio,token_set_ratio
31448,28614,28614,57051,57052,NaN,NaN,0,0,0,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0,100,100,0


In [16]:
prep_df = prep_df.dropna(subset=['question1', 'question2']).reset_index(drop=True)

print("New shape:", prep_df.shape)
print("Missing values:", prep_df.isna().sum().sum())

New shape: (49999, 29)
Missing values: 0


In [17]:
df.sample(50000, random_state=2)

,id,qid1,qid2,question1,question2,is_duplicate,bge_similarity
339499,339499,665522,665523,Why was Cyrus Mistry removed as the Chairman o...,Why did the Tata Sons sacked Cyrus Mistry?,1,0.882079
289521,289521,568878,568879,By what age would you think a man should be ma...,When my wrist is extended I feel a shock and b...,0,0.319808
4665,4665,9325,9326,How would an arbitrageur seek to capitalize gi...,How would an arbitrageur seek to capitalize gi...,0,0.996398
54203,54203,107861,107862,Why did Quora mark my question as incomplete?,Why does Quora detect my question as an incomp...,1,0.881433
132566,132566,262554,91499,What is it like working with Pivotal Labs as a...,What's it like to work at Pivotal Labs?,0,0.907068
...,...,...,...,...,...,...,...
17006,17006,33939,33940,How should I start my preparation for GRE?,How do I start off for GRE?,1,0.956032
368660,368660,721494,721495,What are a few reasons that a completely non r...,How has Sino-US relationship evolved and how w...,0,0.471668
346180,346180,678369,678370,What are important things for people intending...,What are important things for people intending...,0,0.798270
120896,120896,239582,239583,What is economic cost?,What is the economic cost of corruption?,0,0.721592


In [18]:
# Check the variables you currently have
print("prep_df:", prep_df.shape)
print("BGE df:", df.shape)

print("\nBGE columns:")
print(df.columns.tolist())

prep_df: (49999, 29)
BGE df: (404351, 7)

BGE columns:
['id', 'qid1', 'qid2', 'question1', 'question2', 'is_duplicate', 'bge_similarity']


In [19]:
bge_features = df[['id', 'bge_similarity']].copy()

print("BGE feature shape:", bge_features.shape)
print(bge_features.head())

BGE feature shape: (404351, 2)
   id  bge_similarity
0   0        0.922287
1   1        0.693393
2   2        0.826137
3   3        0.505303
4   4        0.552441


In [20]:
bge_features = df[['id', 'bge_similarity']].copy()

print("BGE feature shape:", bge_features.shape)
print(bge_features.head())

BGE feature shape: (404351, 2)
   id  bge_similarity
0   0        0.922287
1   1        0.693393
2   2        0.826137
3   3        0.505303
4   4        0.552441


In [21]:
check = prep_df[['id']].merge(
    bge_features,
    on='id',
    how='left'
)

print("Rows:", len(check))
print("Missing BGE similarities:", check['bge_similarity'].isna().sum())

Rows: 49999
Missing BGE similarities: 0


In [22]:
prep_bge_df = prep_df.merge(
    bge_features,
    on='id',
    how='left'
)

print("Final shape:", prep_bge_df.shape)
print(prep_bge_df.columns.tolist())

Final shape: (49999, 30)
['Unnamed: 0', 'id', 'qid1', 'qid2', 'question1', 'question2', 'is_duplicate', 'q1_len', 'q2_len', 'q1_num_words', 'q2_num_words', 'word_common', 'word_total', 'word_share', 'cwc_min', 'cwc_max', 'csc_min', 'csc_max', 'ctc_min', 'ctc_max', 'last_word_eq', 'first_word_eq', 'abs_len_diff', 'mean_len', 'longest_substr_ratio', 'fuzz_ratio', 'fuzz_partial_ratio', 'token_sort_ratio', 'token_set_ratio', 'bge_similarity']


In [23]:
prep_bge_df.head()

,Unnamed: 0,id,qid1,qid2,question1,question2,is_duplicate,q1_len,q2_len,q1_num_words,...,last_word_eq,first_word_eq,abs_len_diff,mean_len,longest_substr_ratio,fuzz_ratio,fuzz_partial_ratio,token_sort_ratio,token_set_ratio,bge_similarity
0,339499,339499,665522,665523,why was cyrus mistry removed as the chairman o...,why did the tata sons sacked cyrus mistry,1,57,41,11,...,0.0,1.0,3.0,9.5,0.309524,39,46,67,85,0.882079
1,289521,289521,568878,568879,by what age would you think a man should be ma...,when my wrist is extended i feel a shock and b...,0,51,104,11,...,0.0,0.0,11.0,16.5,0.115385,26,50,35,37,0.319808
2,4665,4665,9325,9326,how would an arbitrageur seek to capitalize gi...,how would an arbitrageur seek to capitalize gi...,0,123,122,42,...,1.0,1.0,0.0,24.0,0.593496,100,99,100,99,0.996398
3,54203,54203,107861,107862,why did quora mark my question as incomplete,why does quora detect my question as an incomp...,1,44,59,8,...,0.0,1.0,2.0,9.0,0.355556,74,75,74,89,0.881433
4,132566,132566,262554,91499,what is it like working with pivotal labs as a...,what is it like to work at pivotal labs,0,53,39,11,...,0.0,1.0,2.0,10.0,0.400000,76,85,78,84,0.907068


In [24]:
print("Rows:", len(check))
print("Missing BGE similarities:", check['bge_similarity'].isna().sum())

Rows: 49999
Missing BGE similarities: 0


In [25]:
print(prep_bge_df['bge_similarity'].describe())

count    49999.000000
mean         0.788351
std          0.162072
min          0.182646
25%          0.704523
50%          0.830608
75%          0.909950
max          1.000000
Name: bge_similarity, dtype: float64


In [26]:
print(
    prep_bge_df['is_duplicate'].value_counts()
)

is_duplicate
0    31593
1    18406
Name: count, dtype: int64


In [27]:
engineered_features = [
    'q1_len',
    'q2_len',
    'q1_num_words',
    'q2_num_words',
    'word_common',
    'word_total',
    'word_share',
    'cwc_min',
    'cwc_max',
    'csc_min',
    'csc_max',
    'ctc_min',
    'ctc_max',
    'last_word_eq',
    'first_word_eq',
    'abs_len_diff',
    'mean_len',
    'longest_substr_ratio',
    'fuzz_ratio',
    'fuzz_partial_ratio',
    'token_sort_ratio',
    'token_set_ratio'
]

In [28]:
'bge_similarity'

'bge_similarity'

In [30]:
from sklearn.model_selection import train_test_split

X_bge = prep_bge_df[engineered_features + ['bge_similarity']].values
y = prep_bge_df['is_duplicate'].values

X_bge_train, X_bge_test, y_train, y_test = train_test_split(
    X_bge,
    y,
    test_size=0.2,
    random_state=1
)

print("X_train:", X_bge_train.shape)
print("X_test :", X_bge_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (39999, 23)
X_test : (10000, 23)
y_train: (39999,)
y_test : (10000,)


In [31]:
print(prep_bge_df.shape)
print(X_bge_train.shape)
print(X_bge_test.shape)

(49999, 30)
(39999, 23)
(10000, 23)


In [32]:
from sklearn.linear_model import LogisticRegression

bge_model = LogisticRegression(
    max_iter=1000,
    random_state=1
)

bge_model.fit(X_bge_train, y_train)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=1000, random_state=1)

In [33]:
bge_pred = bge_model.predict(X_bge_test)

bge_prob = bge_model.predict_proba(X_bge_test)[:, 1]

In [39]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("BGE + Engineered Features + LR")
print("=" * 40)

print("Accuracy :", accuracy_score(y_test, bge_pred))
print("Precision:", precision_score(y_test, bge_pred))
print("Recall   :", recall_score(y_test, bge_pred))
print("F1 Score :", f1_score(y_test, bge_pred))
print("ROC-AUC  :", roc_auc_score(y_test, bge_prob))

BGE + Engineered Features + LR
Accuracy : 0.8152
Precision: 0.7409326424870466
Recall   : 0.7515905947441217
F1 Score : 0.7462235649546828
ROC-AUC  : 0.8939383128030666


In [35]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

xgb_bge = XGBClassifier()

xgb_bge.fit(
    X_bge_train,
    y_train
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [36]:
y_pred_bge_xgb = xgb_bge.predict(X_bge_test)

In [37]:
accuracy_bge_xgb = accuracy_score(
    y_test,
    y_pred_bge_xgb
)

print("BGE + Engineered Features + XGBoost")
print("Accuracy:", accuracy_bge_xgb)

BGE + Engineered Features + XGBoost
Accuracy: 0.8301


In [38]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

y_prob_bge_xgb = xgb_bge.predict_proba(X_bge_test)[:, 1]

print("BGE + Engineered Features + XGBoost")
print("=" * 50)

print("Accuracy :", accuracy_score(y_test, y_pred_bge_xgb))
print("Precision:", precision_score(y_test, y_pred_bge_xgb))
print("Recall   :", recall_score(y_test, y_pred_bge_xgb))
print("F1 Score :", f1_score(y_test, y_pred_bge_xgb))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_bge_xgb))

BGE + Engineered Features + XGBoost
Accuracy : 0.8301
Precision: 0.7535733192165167
Recall   : 0.787551867219917
F1 Score : 0.7701880156905181
ROC-AUC  : 0.9115173334806357
